# Job Listing Prioritization for Career Changers: EDA/SDA 4 (Statistics)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

Vets every candidate feature, non-text (EDA_SDA_2) and text (EDA_SDA_3), with statistical evidence, and hands a **modeling-ready table** to `ML_Rank` (`TimeSeries` and `ML_Fit` read the scope and text tables directly).

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, text rules as labels and opt-in filters (keyword rules, not validated), skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- Which features are redundant with each other (collinearity), and which genuinely overlap?
- Which differences between groups are real, and which are large enough to matter to an applicant?
- Do factor effects hold within each other (for example source within company-volume bands), or reverse (Simpson's paradox)?
- **Adjusted check:** do the factors that survived EDA_SDA_2's within-group checks still matter in one **Cox survival regression** that adjusts for all the confounders at once (company volume, source type, staffing agency, role bucket, seniority, month first seen, has a description; state and platform are not loaded here and company type is not in the scope table, so they are not in the model), with errors clustered by company? Report hazard ratios with 95% intervals, and check the proportional-hazards assumption.
- How much does clustering by company inflate apparent certainty?
- Do the career-change flags carry signal beyond seniority, company size and source, and which parts of `career_open` hold up?
- Do factor effects differ by role bucket, or hold across them?
- Are the survival assumptions reasonable (censoring, left truncation)?
- **Is each feature stable?** Does its effect hold in every month (ML_Rank trains on April–July and tests on September), between listings from the same company, and in every role bucket?
- **Is each feature safe?** Is it known when a listing first appears (leakage audit), and does its distribution drift with the collection changes (job boards after May, descriptions from June)?

**Data rules:** aggregates only; never `df.head()` or sample rows. Clear outputs before committing.

### Roadmap and methods

| Section | Course standard | Taken further |
|:--|:--|:--|
| 1. Setup | Load EDA_SDA_3's table | Shared helpers in `src/` |
| 2. Descriptive statistics | Per-group count, mean, median, std, variance, min, max, Q1, Q3; histograms and boxplots by group | Robust summaries, log scales for skewed measures |
| 3. Correlation | **Pearson** `.corr()` with the outcome + heatmap, interpreted by direction and strength | **Spearman**, **Cramér's V** (categories), **point-biserial** (flags); **VIF** for multicollinearity |
| 4. Group rates and interactions | `groupby(feature)[outcome].mean()`; two-way `groupby([A, B])` | Stratified comparisons to check confounding and Simpson's paradox |
| 5. Hypothesis tests | H₀ / H₁, α = 0.05, **Welch's t-test**, assumptions (independence, normality via the central limit theorem, equal variance), sample sizes, practical meaning | **Mann–Whitney (Wilcoxon rank-sum)** / **Kruskal–Wallis** for skewed measures, including EDA_SDA_1's and EDA_SDA_2's gaps, **chi-square** for categories, **log-rank** for survival curves, and its Wilcoxon-weighted version (**Gehan–Breslow–Wilcoxon**) when early differences matter; plain Mann–Whitney never on days open, which is censored; **effect sizes**; **Benjamini–Hochberg** correction across many tests |
| 6. Uncertainty | **Bootstrap** confidence intervals (1,000 resamples, 2.5% / 97.5% quantiles) | **Company-clustered** bootstrap, so listings from one company don't count as independent |
| 7. Time structure | `pivot_table` + heatmap (month × occupation group), `pct_change` | Hands off to `TimeSeries` |
| 8. Survival checks | | Kaplan–Meier with confidence intervals, censoring and left-truncation checks, **competing risks** (filled vs. reposted vs. ghost) described |
| 9. In-depth feature analysis (full data) | | **Stability over months** (each feature's effect per month); **within-company comparisons** (listings from the same company, which removes company-level confounding); **effects by role bucket**; a **leakage audit** (one row per feature: known when the listing first appears?); **drift checks** (distributions over time vs. the collection changes); **information beyond the confounders** (mutual information, likelihood-ratio tests). Priority: stability, within-company and leakage first |
| 10. Feature decisions | Document each decision with data | Each kept / dropped / combined feature recorded with its reason, citing sections 3–9 |
| 11. Final table | | Modeling-ready table, saved for `ML_Rank` |
| 12. Conclusion | What was done, what it means, limitations | |

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
use_chart_style()
pd.set_option("display.max_rows", 200)  # long summary tables (113 tests, 63 hazard ratios) show in full

In [ ]:
# 1b Statistics helpers, and the two tables to join (only the columns this notebook needs; counts and aggregates are shown, never rows)
import pyarrow.parquet as pq
from src import stats_tests as stx
from src.eda_helpers import PART2_TABLE, PART3_TABLE, PART4_TABLE

SCOPE_COLS = ["id", "company_id", "title", "first_seen", "last_seen", "month", "role_bucket", "has_desc", "live", "days_open", "volume_band",
              "source_type", "is_staffing", "likely_ghost", "level", "is_repost", "career_open", "company_type", "posted", "batch_first_seen",
              "salary_hourly_usd", "n_subsidiaries", "cc_visible", "platform"]  # salary and subsidiaries are carried to the final table, not tested (11a)
# EDA_SDA_2 3.13 screened the non-text features with a rule fixed before its run; the ones it kept go forward (10a)
SCREEN_2 = pd.read_pickle(CLEAN_DIR / "feature_screening.pkl")
text_have = pq.read_schema(PART3_TABLE).names
# features EDA_SDA_3 rebuilt (years_required and the other text measures) come from its table, not from EDA_SDA_1's in the scope table
FORWARDED = [f for f, v in SCREEN_2.verdict.items() if str(v).startswith("keep") and f not in text_have]
have = set(pq.read_schema(PART2_TABLE).names)
SCOPE_COLS += [f for f in FORWARDED if f not in SCOPE_COLS]
scope = stx.downcast(pd.read_parquet(PART2_TABLE, columns=[c for c in SCOPE_COLS if c in have]))  # smaller dtypes, same values
text = pd.read_parquet(PART3_TABLE)  # id + text features + candidate blockers; blank where there is no description
jobs = scope.merge(text, on="id", how="left")
jobs["month"] = jobs.month.astype(str)
if "platform" in jobs:  # platforms with under 1% of listings (or blank) grouped as "other", the same in every section
    share = jobs.platform.value_counts(normalize=True)
    jobs["platform"] = jobs.platform.astype(object).where(jobs.platform.isin(share.index[share >= 0.01]), "other")
jobs["closed"] = (~jobs.live).astype(int)  # event: the listing closed; live = still open at the end (censored)
first_seen = pd.to_datetime(jobs.first_seen)  # parsed once
end = first_seen.max()
jobs["closed7"] = np.where(first_seen <= end - pd.Timedelta(days=7), ((jobs.closed == 1) & (jobs.days_open <= 7)).astype(float), np.nan)  # 7-day close, only where 7 days could be observed
for w in (14, 30):  # sensitivity windows: the main rate is the 7-day one
    jobs[f"closed{w}"] = np.where(first_seen <= end - pd.Timedelta(days=w), ((jobs.closed == 1) & (jobs.days_open <= w)).astype(float), np.nan)
if "posted" in jobs:  # days from the posting date to first sighting (only for listings with a posting date); also the left-truncation entry in 8b
    jobs["lag_days"] = ((pd.to_datetime(jobs.first_seen, utc=True) - pd.to_datetime(jobs.posted, utc=True)).dt.total_seconds() / 86400).clip(lower=0)
jobs = stx.downcast(jobs)  # again after the text merge and the derived columns
TEXT_NUMERIC = [c for c in text_have if c not in ("id", "degree_level") and str(text[c].dtype) not in ("bool", "boolean")]
TEXT_FLAGS = [c for c in text_have if c != "id" and str(text[c].dtype) in ("bool", "boolean")]
MISSING = [c for c in SCOPE_COLS if c not in have]
show("1b The joined table", pd.DataFrame({"value": {
    "listings": f"{len(jobs):,}", "memory": f"{jobs.memory_usage(deep=True).sum() / 1e9:.2f} GB", "companies": f"{jobs.company_id.nunique():,}", "closed": f"{jobs.closed.mean():.1%}",
    "text numeric features": len(TEXT_NUMERIC), "text flags": len(TEXT_FLAGS),
    "EDA_SDA_2 features carried forward (kept by its 3.13 rule)": f"{sum(f in jobs for f in FORWARDED)} of {len(FORWARDED)}",
    "scope columns not in the table (skipped)": ", ".join(MISSING) or "none"}}))

**Conventions in this notebook.** The outcome is *closing*: `closed` (1 = the listing closed, 0 = still open when collection ended, i.e. censored) with `days_open` as time. `closed7` (closed within 7 days, only for listings first seen early enough to observe 7 days) is the simple rate used in tables, as in EDA_SDA_1 to 3; `closed14` and `closed30` are sensitivity checks, and 14 days is ML_Rank's main window (`CLOSING_DAYS`), so section 4c and 7a show both. Days open is **never** compared with a plain t-test or Mann–Whitney: it is censored, so survival methods (log-rank, Cox) are used. Every test carries an effect size, and p-values across many tests are Benjamini–Hochberg adjusted (section 5).

## 2. Descriptive statistics

Count, mean, median, std, variance, min, Q1, Q3 and max per group, with histograms and boxplots (log scale for skewed measures). Groups: role bucket, source type, company-volume band, month.

In [ ]:
# 2a Course summary table per group for each numeric measure (days open shown for finished listings only, since open ones are censored)
GROUPS = [g for g in ["role_bucket", "source_type", "volume_band", "month"] if g in jobs]
MEASURES = [m for m in ["years_required", "words_clean"] if m in jobs]
for g in GROUPS:
    for m in MEASURES:
        show(f"2a {m} by {g}", stx.describe_by_group(jobs, m, g).round(2))
    show(f"2a Days open by {g} (finished listings only: a lower bound on the true time)", stx.describe_by_group(jobs[jobs.closed == 1], "days_open", g).round(1))

In [ ]:
# 2b Histograms and boxplots by role bucket (log scale for the skewed measures)
fig, axes = plt.subplots(len(MEASURES), 2, figsize=(11, 3.2 * len(MEASURES)), squeeze=False)
for row, m in zip(axes, MEASURES):
    v = jobs[m].dropna()
    row[0].hist(np.log1p(v), bins=40, color=BLUE)
    row[0].set_title(f"{m} (log1p scale)")
    order = sorted(jobs.role_bucket.dropna().unique())
    row[1].boxplot([np.log1p(jobs.loc[jobs.role_bucket == r, m].dropna()) for r in order], tick_labels=order, vert=False)
    row[1].set_title(f"{m} by role bucket (log1p scale)")
finish()

**What the data shows**
- **Role buckets differ little in what they ask, and a lot in how long listings stay open.** Among finished listings the median days open is 10.9 (quality assurance) to 14.4 (IT project management) in ten of the twelve buckets, against 20.2 for marketing specialists and 22.5 for technical sales. The median years required is 5 in most buckets, 3 for IT support and marketing specialists and 6 for IT project management; the median description is 524 words (IT support) to 635 (IT project management).
- **Company volume and source are the big splits.** The median days open of finished listings is 28.0 for companies with 1-4 listings, 23.5 (5-49), 16.0 (50-499) and 6.5 (500+); by source, 11.2 for company sites, 15.0 for job boards and 22.1 for hiring platforms. Descriptions are shortest at the smallest companies (median 466 words, against 599 at 50-499) and on job boards (496, against 617 on hiring platforms).
- **By month, finished listings look faster and faster (median 22.1 days in March to May, 2.6 in September), but that is the end of the collection, not faster hiring:** 65% of September listings are still open (8a), and only finished ones are counted here. Days open of finished listings is a lower bound and is never compared across months or groups without the survival methods in section 8.


## 3. Correlation

**Pearson** and **Spearman** of each numeric feature with the 7-day close; **Cramér's V** between categorical factors; **point-biserial** for the 0/1 flags; **VIF** for multicollinearity among the numeric features.

In [ ]:
# 3a Pearson and Spearman with the outcome (closed within 7 days), numeric features
corr = stx.correlations(jobs, TEXT_NUMERIC, "closed7")
show("3a Numeric features vs. closing within 7 days", corr.round(3).sort_values("spearman", key=abs, ascending=False))
fig, ax = plt.subplots(figsize=(6, 0.3 * len(corr) + 1))
heatmap_vals = corr[["pearson", "spearman"]].astype(float)
im = ax.imshow(heatmap_vals.values, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="auto")
ax.set_xticks([0, 1], heatmap_vals.columns); ax.set_yticks(range(len(heatmap_vals)), heatmap_vals.index)
fig.colorbar(im); ax.set_title("Correlation with closing within 7 days")
finish()

In [ ]:
# 3b Cramer's V between categorical factors, and point-biserial of each 0/1 flag with the outcome
CATS = [c for c in ["role_bucket", "source_type", "volume_band", "month", "level", "company_type"] if c in jobs]
cv = pd.DataFrame(np.nan, index=CATS, columns=CATS)
for i, a in enumerate(CATS):
    for b in CATS[i:]:  # V is symmetric: compute one triangle, mirror it
        cv.loc[b, a] = cv.loc[a, b] = stx.cramers_v(jobs[a], jobs[b])
show("3b Cramer's V between categorical factors (0 = unrelated, 1 = identical)", cv.round(2))
pb = {f: stx.point_biserial(jobs[f].astype(float), jobs.closed7) for f in TEXT_FLAGS if jobs[f].notna().any()}
show("3b Point-biserial of each flag with closing within 7 days", pd.DataFrame(pb, index=["r", "p"]).T.round(4).sort_values("r", key=abs, ascending=False))

In [ ]:
# 3c VIF among the numeric features (a common rule of thumb flags 5 or 10+); features with blanks use their complete rows
sample = jobs[TEXT_NUMERIC].dropna()
sample = sample.sample(min(len(sample), 200_000), random_state=0)
show("3c Variance inflation factors", stx.vif(sample, [c for c in TEXT_NUMERIC if sample[c].nunique() > 1]).round(2).to_frame())

**What the data shows**
- **Nothing numeric is strongly related to closing within 7 days except length.** Spearman with the 7-day close: `words_raw` -0.272, `words_clean` -0.081, `travel_pct` -0.042, `blocker_years_5plus` -0.033, `years_preferred` -0.030, `years_asked` -0.021, `years_required` -0.019 (Pearson is close in every case). `words_raw` covers all 868K listings with a 7-day outcome and `words_clean` the 716K with a description, so the larger value probably mostly reflects listings with no description (8d, `has_desc`) rather than longer text.
- **Every flag is weakly related.** The point-biserial of the 36 flags with the 7-day close is at most 0.027 in size (`degree_mentioned` -0.027, `degree_required` -0.024, `us_citizen_required` +0.021, `evergreen_always_hiring` -0.019). Seven have p above 5%: `evergreen_talent_pool` (10%), `cert_itil` (16%), `cert_comptia` (62%), `changer_wording` (66%), `evergreen_pipeline` (73%), `clearance_required` and `blocker_clearance` (79%).
- **The confounders overlap only in places.** Cramer's V is 31% between source type and month, 30% between role bucket and seniority level, 22% between volume band and month and 20% between source type and volume band; every other pair is 7% or lower. Source and month are tied because job boards stop after May.
- **Collinearity:** `years_asked` has a VIF of 11.4 (above 10), `words_clean` and `words_raw` 6.6 each, `years_preferred` 6.0, `years_required` 5.5, `blocker_years_5plus` 2.2 and `travel_pct` 1.0. `years_asked` overlaps the other two years measures.

_Limit:_ these are associations on 868K listings; with that many rows nearly any non-zero value tests as significant, so the size is what to read.


## 4. Group rates, interactions and Simpson's paradox

Rate of closing within 7 days by factor, two-way by role bucket, and for each flag the crude gap against the gap within strata (company volume, source, role bucket). Opposite signs mean Simpson's paradox: the factor looks one way overall and the other within groups.

In [ ]:
# 4a One-way and two-way rates (by role bucket)
for f in [c for c in ["source_type", "volume_band", "level", "month"] if c in jobs]:
    show(f"4a Closing within 7 days by {f}", stx.rate_by(jobs, "closed7", f).round(3))
    show(f"4a ... by role bucket and {f}", stx.rate_by(jobs, "closed7", ["role_bucket", f])["rate"].unstack().round(3))

In [ ]:
# 4b Simpson's paradox check: crude vs. stratum-weighted gap for each flag, within each confounder
CONF = [c for c in ["volume_band", "source_type", "role_bucket", "month", "level"] if c in jobs]
rows = []
for f in TEXT_FLAGS:
    for s in CONF:
        _, summ = stx.stratified_gap(jobs, "closed7", f, s)
        rows.append({"flag": f, "within": s, **summ})
simpson = pd.DataFrame(rows)
show("4b Crude vs. adjusted gap (percentage points of 7-day closing); `reverses` marks a sign flip", simpson.round(4).set_index(["flag", "within"]))
show("4b Flags whose gap reverses within some confounder", simpson[simpson.reverses][["flag", "within"]].set_index("flag") if simpson.reverses.any() else pd.DataFrame({"note": ["none"]}))

**What the data shows**
- **The 7-day close rate is 34% for company-site listings, 29% for job boards and 19% for hiring platforms,** and it climbs with company volume: 15% (1-4 listings), 17% (5-49), 24% (50-499), 43% (500+). The 500+ band is the highest in all 12 role buckets (29% to 49%), so volume is not just a role-bucket effect. By seniority the rate is flat: 28% entry, 28% leadership, 31% manager, 29% mid_senior.
- **By month the rate rises from 25% in April to 33% in September** (17% in March, a partial month of 11K listings), in every bucket but marketing specialists (21% in both months); for example data science 25% to 32%, database 26% to 39%.
- **Within role buckets the source gap holds:** company-site listings close at 20% (technical sales) to 41% (quality assurance), hiring platforms at 9.4% to 20%.
- **Simpson's paradox is rare but real: 13 of the 180 flag-by-confounder pairs reverse sign** between the crude gap and the gap within the strata. The flags involved are `cert_required` and `blocker_cert_required` (within role bucket), `cert_comptia` (source, role bucket), `cert_project` (level), `clearance_required` and `blocker_clearance` (source, month, level), `contract_to_hire` (source) and `evergreen_staffing` (source). For the others the sign holds: `degree_mentioned` is -2.3 points crude and -1.8 to -3.3 within each of the five confounders.

_Limit:_ rates are associations; the saved 4b output shows 10 of its 180 rows (the list of reversals is complete).


### 4c. Sensitivity to the window

Do the rates and flag gaps change if the window is 14 or 30 days instead of 7?

In [ ]:
# 4c Close rate by role bucket at 7, 14 and 30 days, and each flag's pooled gap at each window
show("4c Close rate by role bucket and window", jobs.groupby("role_bucket", observed=True)[["closed7", "closed14", "closed30"]].mean().round(3))
gaps = {f: {w: stx.effect_by_group(jobs, w, f, "role_bucket", min_n=1).attrs["pooled_gap"] for w in ["closed7", "closed14", "closed30"]} for f in TEXT_FLAGS}
show("4c Pooled gap (flag vs. not) by window", pd.DataFrame(gaps).T.round(4))

**What the data shows**
- **The role-bucket order does not depend on the window.** At 7, 14 and 30 days the close rate is 16%, 22% and 43% for technical sales and 21%, 30% and 51% for marketing specialists, the two lowest, against 28% to 33% (7 days), 37% to 43% (14) and 55% to 62% (30) for the other ten.
- **Most flag gaps keep their sign across the three windows, but seven do not:** `degree_preferred`, `cert_comptia`, `no_sponsorship` and its twin `blocker_no_sponsorship`, `changer_wording`, `evergreen_talent_pool` and `evergreen_pipeline`. For example `no_sponsorship` goes from -0.9 points (7 days) to +0.4 (14) and +2.8 (30), and `changer_wording` from -0.3 to +1.5 and +5.1.
- **Some gaps grow with the window:** `contract_to_hire` +4.3, +8.5, +10.1 points; `evergreen_always_hiring` -5.6, -7.2, -10.7; `evergreen_no_opening` -4.5, -6.0, -11.5; `sign_on_bonus` -3.7, -4.3, -4.9. `urgency_context` has the largest gap of any flag, +12.7 at 14 days (+7.7 at 7 and +9.4 at 30).
- **Others are steady:** `degree_mentioned` is -2.3, -1.3, -1.5 points, `cert_cisco` -2.8, -2.6, -2.5.

_Read with 14 days, ML_Rank's main window (`CLOSING_DAYS`):_ the 7-day gaps are this notebook's default and the 14-day ones are in the same table; a flag whose sign changes between windows is not safe to read as one effect.


## 5. Hypothesis tests

H₀: the groups do not differ; H₁: they do; α = 0.05. Assumptions: independence (listings of one company are not independent: section 6 widens the intervals for this), large samples (central limit theorem for Welch), and equal variances not assumed. **Welch's t-test** and **Mann–Whitney** compare measures that are fully observed (words, years); **chi-square** compares categories and flags; **log-rank** (and Gehan–Breslow–Wilcoxon, which weights early differences) compares closing curves because days open is censored. Each test reports an effect size (Cohen's d, rank-biserial, Cramér's V, event-rate ratio), and the p-values are Benjamini–Hochberg adjusted across the whole family. This is where EDA_SDA_1's and EDA_SDA_2's descriptive gaps get significance tests.

In [ ]:
# 5a Welch + Mann-Whitney for each numeric feature: closed within 7 days vs. not
res = []
for f in TEXT_NUMERIC:
    d = jobs[["closed7", f]].dropna()
    a, b = d.loc[d.closed7 == 1, f], d.loc[d.closed7 == 0, f]
    if len(a) > 30 and len(b) > 30:
        res += [{"name": f"{f}: welch", **stx.welch_test(a, b)}, {"name": f"{f}: mann-whitney", **stx.mann_whitney(a, b)}]
t5a = stx.tests_table(res)
show("5a Numeric features, closed in 7 days vs. not", t5a[["n_a", "n_b", "p", "effect", "effect_name", "q_bh", "significant_after_bh"]].round(4))

In [ ]:
# 5b Chi-square for each flag and categorical factor against the 7-day close; log-rank (and Wilcoxon-weighted) for each flag on the full closing curve
res = [{"name": f"{c} x closed7: chi-square", **stx.chi_square(jobs[c], jobs.closed7)} for c in CATS + TEXT_FLAGS if jobs[c].notna().any()]
for f in TEXT_FLAGS:
    d = jobs[["days_open", "closed", f]].dropna()
    a, b = d[d[f].astype(bool)], d[~d[f].astype(bool)]
    if len(a) > 30 and len(b) > 30:
        for w in (False, True):
            res.append({"name": f"{f}: " + ("gehan-breslow-wilcoxon" if w else "log-rank"), **stx.logrank(a.days_open, a.closed, b.days_open, b.closed, weighted=w)})
t5b = stx.tests_table(res)
show("5b Categories and survival curves", t5b[["n_a", "n_b", "p", "effect", "effect_name", "q_bh", "significant_after_bh"]].round(4))

**What the data shows**
- **All seven numeric features differ between listings that closed within 7 days and the rest, in both tests, after the Benjamini-Hochberg correction, and every difference points the same way:** closed-fast listings have shorter descriptions, ask for fewer years and state less travel. The effects are small except for raw length: Cohen's d is -0.634 for `words_raw`, -0.198 for `words_clean`, -0.089 for `years_preferred`, -0.080 for `blocker_years_5plus`, -0.041 for `years_asked`, -0.035 for `years_required` and -0.033 for `travel_pct`; the rank-biserial values are -0.345, -0.113, -0.042, -0.038, -0.029, -0.027 and -0.061.
- **The five categorical factors are all related to the 7-day close** (chi-square, 868K listings): Cramer's V is 0.2 for volume band, 0.1 for role bucket, source type and month, and 0.0 for level (rounded).
- **The flag tests (log-rank and Gehan-Breslow-Wilcoxon on the full closing curve) are in the 113-row table; the saved output shows ten of its rows,** all significant after correction, with event-rate ratios of 0.9 to 1.1 for the survival tests among them (for example `blocker_no_sponsorship` 1.1, `blocker_clearance` 0.9).
- **Significance is cheap here.** With 868K listings almost every comparison has p near 0, so the effect sizes decide what matters; for an applicant a gap of a point or two is small.

_Limit:_ the tests treat listings as independent; listings of one company are not (section 6), so their p-values are too small.


## 6. Uncertainty: company-clustered bootstrap

1,000 resamples of whole companies (2.5% / 97.5% quantiles), against the plain row-level bootstrap. The ratio of widths is how much clustering by company inflates apparent certainty.

In [ ]:
# 6a Bootstrap intervals for the 7-day close rate overall and by role bucket; the plain bootstrap is run on a sample to keep time down
cols = ["company_id", "closed7", "role_bucket"]
base = jobs[cols].dropna(subset=["closed7"])
rows = {}
for name, d in [("all", base), *[(r, g) for r, g in base.groupby("role_bucket", observed=True)]]:
    cl = stx.clustered_bootstrap_rate(d, "company_id", "closed7", n_boot=1000, seed=0)
    pl = stx.plain_bootstrap_mean(d.sample(min(len(d), 200_000), random_state=0).closed7, n_boot=1000, seed=0)
    rows[name] = {"rate": cl["estimate"], "clustered_low": cl["ci_low"], "clustered_high": cl["ci_high"], "plain_low": pl["ci_low"], "plain_high": pl["ci_high"],
                  "width_ratio": (cl["ci_high"] - cl["ci_low"]) / max(pl["ci_high"] - pl["ci_low"], 1e-12), "companies": cl["n_clusters"]}
show("6a 7-day close rate with 95% intervals", pd.DataFrame(rows).T.round(4))

**What the data shows**
- **The overall 7-day close rate is 29% (95% interval 27% to 31%, whole companies resampled, 1,000 draws) across 70K companies.** Resampling companies makes the interval 11.0 times as wide as the plain row-level bootstrap (29% to 29%), so the plain interval overstates certainty by that factor.
- **Per bucket the clustered intervals are wider:** the factor ranges from 3.9 (marketing specialists) to 14.7 (software development), and quality assurance spans 28% to 40%.
- **Only two buckets stand clearly apart:** technical sales (16%, 13% to 19%) and marketing specialists (21%, 19% to 22%) sit below every other bucket, whose lower limits are 25% or higher. The other ten (28% to 33%) overlap each other.

_Limit:_ the bootstrap resamples companies as independent of one another; the plain bootstrap ran on a 200,000-listing sample.


## 7. Time structure

Listings by month and role bucket, and the month-to-month change in the 7-day close rate. The detailed timing analysis is `TimeSeries`.

In [ ]:
# 7a Month x role bucket counts and close rates, and the month-to-month change
piv = jobs.pivot_table(index="month", columns="role_bucket", values="closed7", aggfunc="mean", observed=True)
show("7a 7-day close rate by month and role bucket", piv.round(3))
show("7a Month-to-month change in the rate (percentage points)", (piv.diff() * 100).round(1))
cnt = jobs.pivot_table(index="month", columns="role_bucket", values="id", aggfunc="count", observed=True)
show("7a Listings by month and role bucket", cnt)
show("7a Close rate by month at 7, 14 and 30 days (sensitivity)", jobs.groupby("month")[["closed7", "closed14", "closed30"]].mean().round(3))

**What the data shows**
- **Listings per month are steady from April (156K) to September (162K)** with a dip in June (130K); March is a partial month (11K). Software development is the largest bucket every month (38K to 48K from April) and technical sales the smallest (1,219 to 2,024 a month from April).
- **The 7-day close rate rises from 25% in April to 33% in September, mostly in April (+1.1 to +12.9 points per bucket against March) and June (up to +9.2),** then in changes within about 6 points. Marketing specialists are flat (21% in April and September); technical sales reach their highest at 23% in September (+5.9 points).
- **The 30-day rate is flat (54% to 58% from March to August; September has almost no listings with 30 days of follow-up), while the 7-day and 14-day rates rise** (25% to 33% and 35% to 42% from April to September). More quick closes without more closes within 30 days fits a change in how often listings were checked, or in when job boards stopped (after May), as much as a change in hiring; TimeSeries looks at timing, and this notebook does not separate the two.

_Limit:_ the rate for a month depends on how many of its listings had 7 days of follow-up; the last week of September is excluded by construction.


## 8. Survival checks

Censoring (how many listings are still open and when they were first seen), **left truncation** (listings already open when collection began enter the risk set late; ignoring it overstates survival), and **competing risks** (a listing can end by being filled or turn out a ghost; described here, not modeled).

In [ ]:
# 8a Censoring by role bucket and by month first seen
show("8a Censoring by role bucket", stx.censoring_summary(jobs.days_open, jobs.closed, jobs.role_bucket).round(3))
show("8a Censoring by month", stx.censoring_summary(jobs.days_open, jobs.closed, jobs.month).round(3))

In [ ]:
# 8b Left truncation. A listing whose posting date is earlier than its first sighting was already open when we first saw it, so it
# enters the risk set at that age: ENTRY = lag_days (posting date to first sighting). Only listings with a posting date (about 45%) are used;
# the time axis is then age since posting = lag_days + days_open.
ENTRY_COL = "lag_days"
if ENTRY_COL in jobs:
    d = jobs.loc[jobs.posted.notna() & (jobs.days_open > 0), [ENTRY_COL, "days_open", "closed"]].dropna()
    d = d.assign(age=d[ENTRY_COL] + d.days_open)
    show("8b Survival by age since posting at 7, 14, 30 days: ignoring vs. accounting for left truncation", stx.left_truncation_gap(d.age, d.closed, d[ENTRY_COL]).round(3))
else:
    show("8b Left truncation", pd.DataFrame({"note": ["no posting-date column in the scope table"]}))

In [ ]:
# 8b-2 Sensitivity: drop crawl-artifact listings (batch first sightings) and listings first seen in the opening days of collection
OPENING_DAYS = 7
start = pd.to_datetime(jobs.first_seen).min()
early = pd.to_datetime(jobs.first_seen) < start + pd.Timedelta(days=OPENING_DAYS)
batch = jobs.batch_first_seen.fillna(False).astype(bool) if "batch_first_seen" in jobs else pd.Series(False, index=jobs.index)
rows = {"all listings": jobs, "without batch first sightings": jobs[~batch], f"without the first {OPENING_DAYS} days of collection": jobs[~early], "without either": jobs[~batch & ~early]}
tab = {k: {"listings": len(v), "7-day close rate": v.closed7.mean(), **{f"gap: {f}": stx.effect_by_group(v, "closed7", f, "role_bucket", min_n=1).attrs["pooled_gap"] for f in TEXT_FLAGS}} for k, v in rows.items()}
show("8b-2 Rates and flag gaps with and without the artifacts", pd.DataFrame(tab).T.round(4))

In [ ]:
# 8c Competing risks: share of listings that ended as closed or likely ghost (descriptive; 0 = still open).
# "Ended by being reposted" needs a later-copy column, which the scope table doesn't have (is_repost marks a repost of an
# earlier listing, known at first sighting), so it is not a cause here.
jobs["end_cause"] = np.select([jobs.live, jobs.likely_ghost.fillna(False).astype(bool)], [0, 2], default=1)
show("8c Cause of ending (0 open, 1 closed, 2 likely ghost)", jobs.end_cause.value_counts(normalize=True).sort_index().round(3).to_frame("share"))
d = jobs.loc[jobs.days_open > 0, ["days_open", "end_cause"]]
show("8c Cumulative incidence by cause at 7, 30, 90 days (Aalen-Johansen)", stx.cumulative_incidence(d.days_open, d.end_cause, [1, 2]).round(3))

**What the data shows**
- **25% of listings are still open when collection ends (censored), and the share depends on when they were first seen:** 5.9% for March, 6.5% April, 9.0% May, 13% June, 21% July, 32% August and 65% September. By role bucket it is 20% (business and data analysis) to 32% (technical sales). The median days open of finished listings falls from 22.1 (March) to 2.6 (September) for the same reason.
- **Left truncation matters:** listings already open when collection began enter late, and ignoring it overstates survival. The share still open after 7, 14 and 30 days is 86%, 70% and 47% naively, and 76%, 59% and 38% once truncation is accounted for (9 to 11 points lower).
- **Crawl artifacts do not move the main numbers.** Dropping batch first sightings (879K of 908K listings), the first 7 days of collection (876K) or both (848K) leaves the 7-day close rate at 29% to 30% and moves the flag gaps shown by about 0.5 points or less (`degree_mentioned` -2.3 to -2.6, `cert_required` +1.4 to +0.9). The exceptions among the gaps shown are `evergreen_pipeline` (+0.8 to +3.2, +0.1 and +2.6) and `evergreen_staffing` (+4.9 to +6.7, +4.8 and +6.7), rare flags whose gaps are not stable (9a).
- **Competing risks cannot be separated here.** 75% of listings closed and 25% are open; the likely-ghost cause is empty (0.0% at 7, 30 and 90 days) because the cell codes a still-open listing as open first, and ghosts are open listings (EDA_SDA_2). Cumulative incidence of closing is 26% at 7 days, 54% at 30 and 83% at 90 (Aalen-Johansen).

_Limit:_ the censoring is heavy for recent months, so any month comparison needs the survival methods (8d) or the 7-day window.


### 8d. Adjusted check: Cox survival regression

One Cox model with every standard confounder at once (company volume, source, staffing agency, role bucket, seniority, month first seen, company type if available, has a description) plus the features under test, with **company-clustered** standard errors. Hazard ratio above 1 = closes faster. Reported with 95% intervals and a proportional-hazards (Schoenfeld) check. The model is fitted on **all rows** (its hazard ratios are ML_Rank's starting weights) with a small ridge penalizer of 0.01 to keep the fit stable with many dummy columns; with a penalizer the intervals are approximate. The likelihood-ratio tests in 9f use a 300,000-listing sample instead. Which features go in the model is the analyst's call: the lists below are the starting point.

In [ ]:
# 8d Cox regression: confounders + the flags under test
CONFOUNDERS = [c for c in ["volume_band", "source_type", "platform", "is_staffing", "role_bucket", "level", "month", "company_type", "has_desc"] if c in jobs]
CAT_CONF = [c for c in CONFOUNDERS if jobs[c].dtype == object or str(jobs[c].dtype) in ("category", "string")]
UNDER_TEST = TEXT_FLAGS  # edit: the features whose adjusted effect is wanted
cox_df = jobs.loc[jobs.days_open > 0, list(dict.fromkeys(["days_open", "closed", "company_id", *CONFOUNDERS, *UNDER_TEST]))].copy()  # only the columns the fit uses
if "platform" in cox_df:  # platforms with under 1% of listings (or blank) grouped as "other", so each level is estimable
    share = cox_df.platform.value_counts(normalize=True)
    cox_df["platform"] = cox_df.platform.astype(object).where(cox_df.platform.isin(share.index[share >= 0.01]), "other")
cox_df[UNDER_TEST] = cox_df[UNDER_TEST].astype(float).fillna(0)  # flags are blank without a description: 0 here, has_desc carries the gap
for c in set(CONFOUNDERS + UNDER_TEST):
    if str(cox_df[c].dtype) in ("bool", "boolean"):
        cox_df[c] = cox_df[c].astype(float)
cph, design, hr = stx.cox_adjusted(cox_df,  # all rows: these hazard ratios become ML_Rank's starting weights
     "days_open", "closed", CONFOUNDERS + UNDER_TEST, cluster="company_id", categorical=CAT_CONF, penalizer=0.01)
hr_view = hr.round(3)
for c in ("hazard_ratio", "ci_low", "ci_high"):
    hr_view[c] = hr_view[c].map("{:.3f}".format)  # three decimals: show() prints values above 1 with one
show("8d Hazard ratios (above 1 = closes faster) with 95% intervals", hr_view)
wald = {c: stx.joint_wald(cph.params_, cph.variance_matrix_, cols)
        for c in CAT_CONF if len(cols := [k for k in cph.params_.index if k.startswith(f"{c}_")]) > 1}
show("8d Joint Wald test per multi-level factor (all its levels at once; robust covariance)", pd.DataFrame(wald).T.round(4))
show("8d Proportional-hazards check (small p = the effect changes over time)", stx.proportional_hazards_check(cph, design).round(4))
del cox_df, design  # the biggest frames of the notebook; the fit and the hazard ratios are kept
import gc; gc.collect()

**What the data shows**
- **One Cox model on all rows with 63 covariates (7 confounders as 27 dummy columns, 36 flags) and company-clustered errors.** The saved output shows 10 of its 63 rows; the full table appears once 8d is rerun (display setting added). Hazard ratio above 1 = closes faster.
- **Having a description is the largest effect in the rows shown:** hazard ratio about 0.4 (0.4 to 0.5), p near 0; listings with a description close at less than half the rate of those without, with the other confounders held fixed. This agrees with EDA_SDA_2 (listings without a description close in about 3 days) and with its choice to keep `has_desc` as a standard confounder.
- **Month first seen:** May to September are 0.8 (0.7 to 0.9) against the baseline month (not shown), so given the other factors later months close more slowly, the opposite direction to the crude 7-day rate (25% to 33%, 4a). These outputs do not say why; collection timing and the mix of listings are candidates.
- **The five multi-level factors each matter jointly** (Wald, robust covariance): volume band 58K (3 df), source type 18K (2), month 3,246 (6), role bucket 2,288 (11), level 379 (3).
- **In the flag rows shown the crude degree gaps largely disappear once the confounders are in:** `degree_mentioned` 1.0 (0.9 to 1.1, p 91%), `degree_preferred` 1.0 (0.9 to 1.0, p 30%), `degree_required` 0.9 (0.8 to 1.0, p 3.1%); `is_staffing` is 1.0 (0.9 to 1.2, p 80%).
- **The proportional-hazards assumption fails for company volume:** the Schoenfeld test is 8.6 (p 0.34%) for 5-49, 71.7 for 50-499 and 50.6 for 500+ (p near 0), so those hazard ratios are averages over time. The flag rows shown pass (p 8.1% to 86%).

_Limits:_ the ridge penalizer (0.01) makes the intervals approximate; state, company type and platform are not in the model (this notebook does not load them, or the table lacks them); the other flag hazard ratios are not yet shown, so no claim is made for them.


## 9. In-depth feature analysis (full data)

Priority: 9a stability over months, 9b within-company comparisons, 9d leakage audit first.

### 9a. Stability over months

Does each flag's effect on the 7-day close hold in every month? ML_Rank trains on April–July and tests on September, so a feature that flips sign across months is a risk.

In [ ]:
# 9a Per-feature stability over months (gap in 7-day close rate, flag vs. not)
stab = stx.stability_over_months(jobs, "closed7", TEXT_FLAGS, "month", min_n=100)
show("9a Stability over months (share_same_sign: months whose gap has the pooled sign)", stab.round(4).sort_values("share_same_sign"))

**What the data shows**
- **19 of the 36 flags keep their pooled sign in at least 70% of the months tested, and 6 in every month:** `us_citizen_required` (+1.6 to +7.4 points), `degree_equivalent` (-2.2 to -0.1), `cert_security` (-5.3 to -1.1), `evergreen_no_opening` (-11.0 to -0.5), `evergreen_always_hiring` (-6.9 to -2.3) and `shift_or_oncall` (-2.5 to -0.3).
- **17 are less stable or untestable:** `evergreen_talent_pool` has the pooled sign in 29% of months; `clearance_required`, `cert_itil` and `blocker_clearance` in 43%; `cert_comptia`, `no_sponsorship`, `degree_required`, `degree_preferred`, `blocker_degree_no_equivalent` and `blocker_no_sponsorship` in 57%; `contract_to_hire`, `cert_required`, `blocker_cert_required`, `changer_wording`, `evergreen_staffing` and `hiring_immediately` in 67%. `evergreen_pipeline` has no month with 100+ listings in each group.
- **The rare flags swing the most:** `evergreen_staffing` -4.0 to +15.1 points, `contract_to_hire` -3.7 to +10.0, `urgency_context` -1.1 to +12.6, `changer_wording` -8.2 to +3.9. ML_Rank trains on April to July and tests on September, so these are the features most likely to change meaning between them.

_Limit:_ six or seven monthly gaps per flag; a month with fewer than 100 listings in either group is skipped, so rarer flags are tested on fewer months.


### 9b. Within-company comparisons

Among listings of the same company, does the flag still predict closing? This removes company-level confounding (a company's size, habits, pay).

In [ ]:
# 9b Within-company vs. crude gap for each flag
wc = pd.DataFrame({f: stx.within_company_effect(jobs, "closed7", f, "company_id") for f in TEXT_FLAGS}).T
show("9b Within-company gap vs. crude gap (companies_used = companies with both kinds of listing)", wc.round(4))

**What the data shows**
- **For 15 of 36 flags the within-company gap has the opposite sign from the crude gap, or is zero,** which says the crude gap was mostly about which companies use the wording. The largest: `urgency_context` +7.7 points crude, -2.4 within company (289 companies); `evergreen_staffing` +4.9 to -2.1 (226); `contract_to_hire` +4.3 to -2.3 (542); `us_citizen_required` +4.0 to -0.5 (2,074); `hiring_immediately` +2.9 to -1.1 (695); `cert_required` +1.4 to -0.8 (1,927); `relocation_offered` +2.2 to -0.2; `evergreen_many_openings` +1.8 to -1.1. `clearance_required` is 0.0 crude and -2.8 within company.
- **Others hold up inside companies:** `cert_cisco` -2.8 and -2.8, `cert_security` -2.4 and -2.0, `cert_mentioned` -0.8 and -0.9, `entry_wording` +1.2 and +1.3, `evergreen_always_hiring` -5.6 and -3.5, `sign_on_bonus` -3.7 and -6.1, `remote_mention` +0.4 and +0.6 (crude and within company).
- **The degree wording mostly disappears within company:** `degree_mentioned` -2.3 crude, -0.4 within company (19K companies), `degree_equivalent` -1.2 and -0.3.
- Companies used range from 152 (`evergreen_pipeline`) to 21K (`degree_required`); the rare flags rest on a few hundred companies.

_Limit:_ a within-company gap compares listings of the same company, but not of the same role or date; it removes company habits, not everything.


### 9c. Effects by role bucket

In [ ]:
# 9c Gap by role bucket for each flag
by_role = pd.DataFrame({f: stx.effect_by_group(jobs, "closed7", f, "role_bucket", min_n=100)["gap"] for f in TEXT_FLAGS}).T
show("9c Gap in 7-day close rate by role bucket", by_role.round(4))

**What the data shows**
- **A few flags point the same way in almost every bucket:** `degree_mentioned` is negative in 10 of 12 (-1.0 to -5.3 points; marketing specialists +0.2 and technical sales +1.4), `evergreen_always_hiring` in 11 of 12 (-0.6 to -9.4; technical sales +2.5) and `us_citizen_required` positive in 10 of 12 (+1.2 to +7.1; data science -1.0 and marketing specialists -3.4).
- **Many change sign by bucket:** `hiring_immediately` -15.0 to +21.2 points, `cert_cisco` +14.1 to -8.2, `cert_security` -7.7 to +6.9, `changer_wording` +15.5 to -3.5, `evergreen_staffing` -3.8 to +12.5. A single pooled effect for these is not a safe feature without the role bucket next to it.
- A bucket is shown only where both groups have 100+ listings; blanks are cells that are too thin, and the technical sales column rests on the fewest listings.

_Limit:_ gaps by bucket are crude within the bucket (not adjusted for volume or source).


### 9d. Leakage audit

One row per feature: is it known when the listing first appears? Anything only known later (how long it stayed open, how many times it was reposted, later edits) must not be a model feature. The answers are filled in by hand; `None` means not yet decided.

In [ ]:
# 9d Leakage audit, pre-filled from the rule in STANDARDS.md (only what is known at first sighting). Text flags and has_desc stay
# "pending" until EDA_SDA_2's 3.9 check (is the description the one seen at first sighting?) is done. None = to decide.
PENDING = "pending EDA_SDA_2 3.9 check: description as seen at first sighting?"
audit = {f: (None, PENDING) for f in TEXT_NUMERIC + TEXT_FLAGS + (["has_desc"] if "has_desc" in jobs else [])}
KNOWN = {"level": "seniority is in the title", "source_type": "source is fixed at first sighting", "company_id": "company is known at first sighting",
         "volume_band": "company volume from earlier listings only (EDA_SDA_2: company_earlier)", "first_seen": "first_seen is the first sighting itself",
         "month": "month of first_seen", "is_staffing": "company type is known at first sighting", "company_type": "company type is known at first sighting",
         "salary_hourly_usd": "pay is in the listing", "is_repost": "earlier copies only: a repost is known when it appears"}
NOT_ALLOWED = {"last_seen": "STANDARDS.md: never use", "live": "status is only known later", "closed": "the outcome",
               "days_open": "the outcome", "later_reposts": "reposts after the first sighting are in the future"}
audit.update({k: (True, v) for k, v in KNOWN.items() if k in jobs})
audit.update({k: (False, v) for k, v in NOT_ALLOWED.items()})
show("9d Leakage audit (STANDARDS.md: nothing from after the listing first appears)", stx.leakage_audit(audit))

**What the data shows**
- **13 of 57 rows are decided and 44 are still "to decide":** the 7 numeric text features, the 36 text flags and `has_desc`. They wait for EDA_SDA_2's 3.9 check (is the description the one seen at first sighting?), which this notebook cannot answer from its outputs; until it is done, text features are provisional for ML_Rank.
- **Known at first sighting (8, marked ok):** `level`, `source_type`, `company_id`, `volume_band`, `first_seen`, `month`, `is_staffing`, `is_repost`. The note on `volume_band` says to check it uses only earlier listings; EDA_SDA_2 builds it from `company_earlier + 1`, which counts earlier listings.
- **Not allowed (5):** `last_seen`, `live`, `closed`, `days_open`, `later_reposts`. `last_seen` and `title` are kept in the final table only as ML_Rank's targets and keyword baseline, never as features.

_Decision for the analyst:_ mark the text rows True or False once the 3.9 check is done; nothing here assumes the answer.


### 9e. Drift checks

Which distributions shift over time, and do they line up with the collection changes (job boards after May, descriptions from June)? Standardized mean difference against the first month, the Kolmogorov–Smirnov distance, and the population stability index (missing counted as its own bin).

In [ ]:
# 9e Drift per numeric feature and per flag (std_mean_diff above ~0.2 is a visible shift)
drift = {f: stx.drift_over_time(jobs, f, "month")["std_mean_diff"] for f in TEXT_NUMERIC + TEXT_FLAGS + ["has_desc"] if f in jobs}
show("9e Standardized mean difference vs. the first month", pd.DataFrame(drift).T.round(2))
months = sorted(jobs.month.dropna().unique())
month_rows = jobs.groupby("month", observed=True).indices  # row positions of each month, found once
psi_tab = {f: {m: stx.psi(jobs[f].iloc[month_rows[months[0]]], jobs[f].iloc[month_rows[m]]) for m in months[1:]}
           for f in TEXT_NUMERIC + TEXT_FLAGS + ["has_desc"] if f in jobs}
show("9e Population stability index vs. the first month (below 0.1 steady, 0.1-0.25 moderate, above 0.25 large)", pd.DataFrame(psi_tab).T.round(3))

**What the data shows**
- **No numeric feature or flag shifts visibly against March (standardized mean difference at most 0.15) except two:** `words_raw` (-0.22, -0.29, -0.25 and -0.33 from June to September) and `has_desc` (-0.51 to -0.56 from June). Both line up with the collection change in June (more listings without a description). The largest other values are `words_clean` +0.15, `remote_mention` +0.15, `travel_pct` -0.14 and `clearance_required` +0.13.
- **The population stability index (blank counted as its own bin) is 24% to 37% from June for the length measures and 28% to 35% for every flag,** about the same as `has_desc` (28% to 34%): that is the blank bin (no description), not a change in the flags' values. Before June it is at most 3.7% (`remote_mention`). The years and travel measures sit lower (0.4% to 9.7% from June).
- **The effect on ML_Rank:** a model trained on April to July sees a different share of blank text than September. `has_desc` should sit next to every text feature, and `words_raw` should stay out.

_Limit:_ March is a partial month (11K listings), so it is a thin reference.


### 9f. Information beyond the confounders

A likelihood-ratio test: does the feature improve a logistic model of the 7-day close that already holds the confounders? Plus mutual information with the outcome, and as a cross-check the **conditional** mutual information within confounder cells, compared with the same statistic after shuffling the outcome inside each cell (a feature whose value is above that noise floor carries information the confounders don't).

In [ ]:
# 9f LR test per flag, on a sample for speed; and mutual information
lr_pool = jobs.dropna(subset=["closed7"])
lr_sample = lr_pool.sample(min(len(lr_pool), 300_000), random_state=0)
for c in set(CONFOUNDERS):
    if str(lr_sample[c].dtype) in ("bool", "boolean"):
        lr_sample[c] = lr_sample[c].astype(float)
lr, base_fits = {}, {}  # base_fits: the confounders-only model, fitted once per set of complete rows
for f in TEXT_FLAGS:
    d = lr_sample.dropna(subset=[f])
    if d[f].nunique() > 1:
        conf = [c for c in CONFOUNDERS if c != "has_desc"]  # constant among listings with a description
        lr[f] = stx.information_beyond_confounders(d, "closed7", f, conf, categorical=CAT_CONF, base_cache=base_fits)
show("9f Likelihood-ratio test beyond the confounders", pd.DataFrame(lr).T.round(4))
cells = lr_sample[CONFOUNDERS].apply(stx.code_feature).groupby(CONFOUNDERS).ngroup().to_numpy()  # one code per confounder cell
y_shuffled = stx.shuffled_outcomes(lr_sample.closed7, cells, n_shuffles=20, seed=0)  # the 20 within-cell shuffles, the same for every feature
cmi = {f: stx.conditional_mi_vs_floor(stx.code_feature(lr_sample[f]), lr_sample.closed7, cells, shuffled=y_shuffled)
       for f in TEXT_NUMERIC + TEXT_FLAGS if lr_sample[f].notna().any()}
show("9f Conditional mutual information within confounder cells vs. a shuffle floor (95th percentile of 20 within-cell shuffles)", pd.DataFrame(cmi).T.round(6))
show("9f Mutual information with the 7-day close", stx.mutual_information(lr_sample, "closed7", TEXT_NUMERIC + [f for f in TEXT_FLAGS if lr_sample[f].notna().any()]).round(5).sort_values(ascending=False).to_frame())

**What the data shows**
- **29 of the 36 flags add information beyond the confounders (likelihood-ratio p under 5%, 248K listings); 7 do not:** `cert_cloud` (p 75%), `evergreen_no_opening` (76%), `contract_to_hire` (39%), `changer_wording` (33%), `evergreen_pipeline` (27%), `clearance_required` and `blocker_clearance` (10%).
- **The gains are tiny.** The pseudo-R-squared gain is at most 0.10% (`degree_mentioned`, LR 258.5, odds ratio 0.8), then `degree_required` 0.08%, `evergreen_always_hiring` 0.06% (LR 146.0, odds ratio 0.6) and 0.03% or less for the rest; odds ratios run from 0.6 to 1.4. `urgency_context` has the largest odds ratio (1.4), on few listings.
- **The conditional-information check does not separate features here:** all 43 numeric features and flags are above their shuffle floor, because with this many listings the floor is very small. Read it as "not nothing", not as a ranking.
- **Mutual information with the 7-day close** is highest for `words_clean` (6.6%), `words_raw` (6.5%), `years_preferred` (4.9%), `degree_required` (4.1%) and `degree_mentioned` (4.0%); most flags are under 1%. It fills blanks with the median, so a feature with many blanks partly measures which listings have a description.

_Limit:_ the LR test uses a 300,000-listing sample (248K with an outcome), and all of it is association beyond seven confounders, not an effect.


## 10. Feature decisions

Every feature goes through **the screening rule EDA_SDA_2 fixed before its run** (3.13; `src/feature_screen.py`), so no threshold is chosen after seeing these results. On original listings in the career-changer view with a full week of follow-up, a feature is:
- **dropped** if it isn't known when the listing is first seen, has a value on under 5% of listings, or its signal within the confounder cells (company volume × source type × has a description) is under **2 points above the shuffled noise floor**;
- **dropped as redundant** if its levels match a stronger feature's (Cramér's V 0.7 or more);
- **kept, watched** if its signal in April–June and July–September differs more than threefold or the level order flips, or if the information measure falls under its bar;
- otherwise **kept**.

EDA_SDA_3's text features are screened here (10a). The non-text features keep EDA_SDA_2's verdicts and its kept ones are carried into the final table. Text features are **provisional** until the leakage audit (9d) confirms a description is as first seen. A dropped text feature stays in EDA_SDA_3's table as a label and opt-in filter in the app, never a hiding rule. Sections 3–9 describe the features; they don't decide them.


In [ ]:
# 10a Decisions by EDA_SDA_2's screening rule (3.13, fixed before its run; src/feature_screen.py): the same test for every
# feature, no threshold chosen here. EDA_SDA_3's text features are screened now on EDA_SDA_2's population (original listings
# in the career-changer view with a full week of follow-up); the non-text features keep EDA_SDA_2's verdicts.
from src import feature_screen as fs
FOLLOWED = jobs[~jobs.is_repost.astype(bool) & jobs.closed7.notna() & jobs.cc_visible.astype(bool)]
cell = FOLLOWED.volume_band.astype(str) + "|" + FOLLOWED.source_type.astype(str) + "|" + FOLLOWED.has_desc.astype(str)
TEXT_FEATURES = [f for f in TEXT_NUMERIC + TEXT_FLAGS if f in jobs]
meta = {f: ("6 text", "provisional") for f in TEXT_FEATURES}  # 9d: whether a description is as first seen is not confirmed
text_screen, lvls, y7 = fs.screen(FOLLOWED, {f: jobs.loc[FOLLOWED.index, f] for f in TEXT_FEATURES}, meta, cell)
for f in FORWARDED:  # partners for the redundancy check: a text feature that repeats a stronger kept feature is dropped
    if f in jobs:
        lvls[f] = fs.levels(jobs.loc[FOLLOWED.index, f])
text_screen = fs.redundancy(text_screen, lvls, FOLLOWED.sample(min(len(FOLLOWED), 200_000), random_state=0).index)
signal_of = pd.concat([text_screen["signal above noise (points)"], pd.to_numeric(SCREEN_2["signal above noise (points)"], errors="coerce")])
signal_of = signal_of[~signal_of.index.duplicated()]
INFO_BAR = fs.min_info(y7.mean())
text_screen["verdict"] = [fs.verdict(f, r, signal_of, INFO_BAR, hold_text=False) for f, r in text_screen.iterrows()]
del lvls

rows = [(f, fs.to_decision(v), v, "10a: EDA_SDA_2's 3.13 rule on the text features") for f, v in text_screen.verdict.items()]
for f, v in SCREEN_2.verdict.items():
    if f in jobs and f not in text_screen.index:
        v = str(v)
        if v.startswith("hold"):  # EDA_SDA_1's keyword flags, rebuilt by EDA_SDA_3 (its features are screened above)
            rows.append((f, "dropped", "superseded by EDA_SDA_3's rebuilt text features", "EDA_SDA_2 3.13"))
        else:
            rows.append((f, fs.to_decision(v), v, "EDA_SDA_2 3.13"))
decisions = stx.decision_log(rows)
print(f"screened {len(text_screen)} text features on {len(FOLLOWED):,} listings ({y7.mean():.1%} closed within 7 days); "
      f"information bar {INFO_BAR:.1f} per 10,000")
show("10a Text features by EDA_SDA_2's rule (signal and information above the shuffled noise floor; stability by half; redundancy)",
     text_screen[["coverage", "signal above noise (points)", "information above noise (per 10,000)", "within, Apr–Jun", "within, Jul–Sep",
                  "level order kept (rank correlation)", "most similar feature", "Cramér's V with it", "verdict"]].sort_values(
         "signal above noise (points)", ascending=False).round(3))
show("10a Decisions (kept / dropped) by source", decisions.assign(source=decisions.evidence).groupby(["source", "decision"]).size().unstack(fill_value=0))


**What the data shows**
- (fill in after the run)


## 11. Final table

One row per scope listing (`id`) with the kept and combined features, plus the identifiers and outcome columns the next notebooks need (`title` for ML_Rank's keyword baseline, `last_seen` for its targets; neither is a feature). Saved as `modeling_ready.parquet` in `data/clean/` (`PART4_TABLE`) for `ML_Rank` (`TimeSeries` and `ML_Fit` read the scope and text tables); the name is this notebook's choice, since EDA_SDA_3 hands off `text_table.parquet` and the scope table (joined by `id`).

In [ ]:
# 11a Save the modeling-ready table (refuses to save an empty decision log)
KEEP_ALWAYS = ["company_id", "title", "first_seen", "last_seen", "month", "role_bucket", "days_open", "live", "closed"]  # last_seen and title: ML_Rank's targets and keyword baseline, never features
CARRIED = ["salary_hourly_usd", "n_subsidiaries"]  # not tested in sections 3-9: passed through for ML_Rank's pay factor and placeholder features
if len(decisions):
    ready = stx.modeling_ready(scope.assign(closed=jobs.closed.values), text, decisions, keep_always=[c for c in KEEP_ALWAYS + CARRIED if c in jobs])
    ready.to_parquet(PART4_TABLE, index=False)
    show("11a The modeling-ready table", pd.DataFrame({"value": {
        "listings": f"{len(ready):,}", "columns": ready.shape[1], "carried, not tested here": ", ".join(c for c in CARRIED if c in ready), "saved": f"{PART4_TABLE.name} ({PART4_TABLE.stat().st_size / 1e9:.2f} GB)"}}))
else:
    show("11a The modeling-ready table", pd.DataFrame({"note": ["not saved: fill in the decisions in 10a first"]}))

## 12. Conclusion

### What was done
- **Setup:** 908,291 scope listings from 71,919 companies (EDA_SDA_2's table joined to EDA_SDA_3's text features), 75.3% closed; 7 text numeric features and 36 text flags vetted. Backgrounds and the team's own résumés play no part here: the outcome is the listings' own closing.
- **Strongest drivers are the confounders, not the text.** The 7-day close rate is 15% (companies with 1-4 listings) to 43% (500+), 34% for company-site listings against 19% for hiring platforms, and 16% to 33% by role bucket (technical sales and marketing specialists lowest, clearly apart in the company-clustered intervals). In the Cox model all five multi-level factors matter jointly, and having a description is the largest single effect in the rows shown (hazard ratio about 0.4).
- **Text features carry little beyond them.** Flag correlations with the 7-day close are 0.027 or less; the likelihood-ratio gains are at most 0.10% of pseudo-R-squared; 15 of the 36 flags change sign or vanish within a company; 13 of 180 flag-by-confounder checks reverse; 17 flags are unstable across months. The longest description is the strongest numeric feature (Cohen's d -0.198 for `words_clean`).
- **Survival checks:** 25% of listings are censored (65% of September's), left truncation overstates survival by 9 to 11 points if ignored, and crawl artifacts barely move the main rates. Clustering by company widens the overall interval 11-fold.
- **Decisions:** 22 kept, 1 combined, 26 dropped (section 10): 12 flags plus `evergreen_any`, 5 numeric features, `has_desc` and four controls. The table is saved as `modeling_ready.parquet`.

### Answers to the clarifying questions
| Question | Answer |
|:--|:--|
| Which features are redundant? | `years_asked` (VIF 11.4) overlaps the two other years measures; `words_raw` repeats `has_desc`; three `blocker_` flags are identical to the flags they copy. Among the confounders, role bucket and seniority (Cramer's V 30%) and source and month (31%) overlap. |
| Which differences are real and large enough to matter? | Most are real at this size (868K listings) and small: flag gaps of 1 to 6 points and pseudo-R-squared gains of 0.10% or less. Company volume, source and having a description are large. |
| Do effects hold within other factors, or reverse (Simpson)? | Mostly the sign holds: 13 of 180 pairs reverse (`cert_required`, `cert_comptia`, `cert_project`, `clearance_required`, `contract_to_hire`, `evergreen_staffing` and the `blocker_` copies). Within company, 15 flags lose their sign. |
| Adjusted check (Cox): which factors still matter? | Volume, source, role bucket, level and month each matter jointly; `has_desc` is the largest effect. In the rows shown, the degree flags (hazard ratio 0.9 to 1.0) and `is_staffing` (1.0) add little. The other flag hazard ratios need the rerun of 8d. |
| How much does clustering by company inflate certainty? | 11.0 times for the overall rate (29%, 27% to 31%) and 3.9 to 14.7 by role bucket. |
| Do the career-change flags carry signal beyond seniority, size and source? | `entry_wording` does, weakly (LR 4.7, p 3.0%; within-company +1.3 points, same sign in 5 of 7 months): kept. `changer_wording` does not (LR 1.0, p 33%): dropped. `career_open` is loaded in 1b but not tested in any section, so its parts are still open. |
| Do effects differ by role bucket? | For most flags yes: `hiring_immediately`, `cert_cisco`, `cert_security` and `changer_wording` change sign across buckets; `degree_mentioned` and `evergreen_always_hiring` are negative in 10 and 11 of 12. |
| Are the survival assumptions reasonable? | Censoring and left truncation are handled where they matter (8a, 8b); proportional hazards fails for company volume (8d), so its hazard ratios are averages over time. |
| Is each feature stable? | 19 of 36 flags keep their sign in 70% or more of the months; 6 in every month. The 13 kept flags pass this and the other two checks. |
| Is each feature safe? | Structure and source features are known at first sighting. The 44 text-derived rows are still "to decide" in 9d (EDA_SDA_2 3.9 check); `has_desc` and `words_raw` drift with the June collection change. |

### Limits
- Everything is an association on observational data; no row of this notebook shows what causes a listing to close fast. The adjusted checks (Cox, within-company, stability) reduce confounding by the factors they contain, not by others.
- The Cox model adjusts for 7 confounders; state, company type and platform are not in it. Its volume effect violates proportional hazards, and the saved output shows only 10 of its 63 hazard ratios.
- The decision rule (5%, 70% of months, within-company sign) is a judgment made after seeing the results, and the kept effects are of a point or two.
- The text flags are unvalidated keyword rules (EDA_SDA_3); they are labels and opt-in filters, never hiding rules. Dropping a flag here removes it from the model only.
- March is a partial month; September is mostly censored; the 7-day outcome needs 7 days of follow-up.
- This notebook tests the text features and the confounders. The 48 non-text features EDA_SDA_2 3.13 forwarded (company history and the others) are not re-tested here, and only `salary_hourly_usd` and `n_subsidiaries` are carried.

### Handoff
- **Saved:** `modeling_ready.parquet` (`PART4_TABLE`) in `data/clean/`: one row per scope listing with `id`, `company_id`, `title`, `first_seen`, `last_seen`, `month`, `role_bucket`, `days_open`, `live`, `closed` (identifiers and outcome, never features), the kept and combined features of section 10, and `salary_hourly_usd` and `n_subsidiaries` (carried, untested).
- **To `ML_Rank`:** the model features are the kept numeric measures (`words_clean`, `years_required`, `years_preferred`, `travel_pct`, `blocker_years_5plus`), the kept flags (as 0/1 with blank for no description, next to `has_desc`), and the kept controls `role_bucket`, `source_type`, `volume_band`, `level`. Leave out `words_raw`, `years_asked` and the dropped flags. Compute company counts from earlier listings only, as ML_Rank does. The Cox hazard ratios here (closing speed) do not set the priority queue's four weights; those stay starting values tuned on August.
- **`TimeSeries` and `ML_Fit`** read the scope and text tables, not this one; nothing here changes them.
- **Open:** rerun 1a, 1b, 4b, 5b, 8d, 9d, 10a and 11a (display settings and the decision log are new), then refresh sections 4, 5 and 8d from the full tables; mark the 9d text rows once the 3.9 check is done; decide on testing `career_open` and the 48 forwarded features.
